# Initial-margin CVaR MKABSDE

This notebook displays the retained call and butterfly campaigns. It reads the generated JSON files and contains no solver implementation.

## Complete retraining command

Run from the repository root:

```bash
python -m initial_margin.run_campaign \
  --product all \
  --steps 50 100 200 \
  --seeds 42 43 44 45 46 47 48 49 50 51 \
  --paths 200000 \
  --degree 3 \
  --margin-period 0.02 \
  --device cuda
```

Regenerate the same-grid deterministic MKABSDE reference with:

```bash
python -m initial_margin.deterministic_reference \
  --product all \
  --steps 50 100 200 \
  --space-steps 6000 \
  --expectation-nodes 24 \
  --output initial_margin/results/deterministic_reference.json
```

The paper defaults use ten independent seeds, 24 Gauss--Legendre nodes for the call tail, and 64 nodes for the complete butterfly law.

In [1]:
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('repository root not found')
sys.path.insert(0, str(root))
from initial_margin.evaluation import ARTICLE_REFERENCE_VALUES

result_root = root / 'initial_margin' / 'results'
products = ('call', 'butterfly')
reference_payload = json.loads((result_root / 'deterministic_reference.json').read_text())
direct_references = {(row['product'], row['time_steps']): row for row in reference_payload['rows']}

## Initial-margin-induced price corrections

The direct column targets the anticipated equation; the non-linear and linear columns target the two proxy BSDEs. Every displayed number is $100\,(V_0-V_0^{BS})$, the price correction induced by funding initial margin, reported as the mean plus or minus the sample standard deviation across seeds. It is neither the total price nor the CVaR collateral itself.

In [2]:
for product in products:
    payload = json.loads((result_root / f'atm_{product}_experiment.json').read_text())
    base_price = payload['rows'][0]['metrics']['black_scholes_no_im']['price']
    article = ARTICLE_REFERENCE_VALUES[product]
    lines = [
        f'### ATM {product}',
        '',
        '| $N$ | direct MKABSDE correction | deterministic MKABSDE reference | non-linear proxy correction | linear proxy correction |',
        '|---:|---:|---:|---:|---:|',
    ]
    for row in payload['summary']:
        lines.append(
            f"| {row['time_steps']} | {100 * row['direct_mkabsde_price_correction_vs_black_scholes_mean']:.6f} ± {100 * row['direct_mkabsde_price_correction_vs_black_scholes_std']:.2e} | "
            f"{100 * direct_references[(product, row['time_steps'])]['price_correction']:.6f} | "
            f"{100 * row['nonlinear_proxy_price_correction_vs_black_scholes_mean']:.6f} ± {100 * row['nonlinear_proxy_price_correction_vs_black_scholes_std']:.2e} | "
            f"{100 * row['linear_proxy_price_correction_vs_black_scholes_mean']:.6f} ± {100 * row['linear_proxy_price_correction_vs_black_scholes_std']:.2e} |"
        )
    lines.append(
        f"| Published proxy price converted to correction | — | — | {100 * (article['nonlinear_proxy_price'] - base_price):.6f} | {100 * (article['linear_proxy_price'] - base_price):.6f} |"
    )
    display(Markdown('\n'.join(lines)))

### ATM call

| $N$ | direct MKABSDE correction | deterministic MKABSDE reference | non-linear proxy correction | linear proxy correction |
|---:|---:|---:|---:|---:|
| 50 | 1.873583 ± 2.79e-03 | 1.873235 | 2.197847 ± 2.85e-03 | 2.190344 ± 2.84e-03 |
| 100 | 1.869610 ± 1.95e-03 | 1.868720 | 2.192509 ± 2.02e-03 | 2.184971 ± 2.01e-03 |
| 200 | 1.866482 ± 2.64e-03 | 1.866000 | 2.188668 ± 2.84e-03 | 2.181119 ± 2.84e-03 |
| Published proxy price converted to correction | — | — | 2.178830 | 2.298830 |

### ATM butterfly

| $N$ | direct MKABSDE correction | deterministic MKABSDE reference | non-linear proxy correction | linear proxy correction |
|---:|---:|---:|---:|---:|
| 50 | 0.236261 ± 2.00e-04 | 0.236936 | 0.223998 ± 1.41e-04 | 0.223675 ± 1.41e-04 |
| 100 | 0.238475 ± 1.47e-04 | 0.239110 | 0.226270 ± 1.54e-04 | 0.225937 ± 1.54e-04 |
| 200 | 0.239264 ± 2.31e-04 | 0.239904 | 0.227161 ± 1.36e-04 | 0.226823 ± 1.36e-04 |
| Published proxy price converted to correction | — | — | 0.216529 | 0.216529 |

## Interpretation

The direct Monte Carlo solver and `python -m initial_margin.deterministic_reference` implement the same time grid, Black--Scholes difference formulation, implicit update, and Gauss--Legendre CVaR rule. Only the conditional functions are evaluated differently: Hermite regressions on simulated paths versus Gauss--Hermite integration and interpolation on a log-spot grid. Their distance therefore diagnoses Monte Carlo and regression effects at fixed time grid, not the time-discretization error. The proxy row uses only the values reported in the source article; no additional non-reproducible proxy reference is retained. The sample standard deviations measure combined Monte Carlo and regression variability, not regression error alone.